# 05. Grid DP

The state is a **cell** `dp[r][c]`. Usually you can move only right/down (or down/diagonal), so a cell depends on its **top and left** neighbours.

**Learning order:** UNIQUE PATHS → OBSTACLES → MIN PATH SUM → TRIANGLE → MAXIMAL SQUARE → FALLING PATH → LONGEST INCREASING PATH → DUNGEON GAME → INTERVIEW THINKING

## 1. Unique Paths

Robot at the top-left, moves right or down, reaches the bottom-right. `dp[r][c] = dp[r-1][c] + dp[r][c-1]`. First row and first column are `1` (only one way). One row array is enough: `dp[c] += dp[c-1]`.

In [1]:
def unique_paths(m, n):
    dp = [1] * n
    for _ in range(1, m):
        for c in range(1, n):
            dp[c] += dp[c - 1]
    return dp[-1]

print(unique_paths(3, 7), unique_paths(3, 2))   # 28 3

def unique_paths_obstacles(grid):
    n = len(grid[0])
    dp = [0] * n
    dp[0] = 1 if grid[0][0] == 0 else 0
    for row in grid:
        for c in range(n):
            if row[c] == 1:
                dp[c] = 0                           # obstacle: no path through this cell
            elif c > 0:
                dp[c] += dp[c - 1]
    return dp[-1]

print(unique_paths_obstacles([[0, 0, 0], [0, 1, 0], [0, 0, 0]]), unique_paths_obstacles([[0, 1], [0, 0]]))   # 2 1

28 3
2 1


## 2. Minimum Path Sum

`dp[r][c] = grid[r][c] + min(dp[r-1][c], dp[r][c-1])`. Handle the first row/column separately (only one direction). We can overwrite the grid or use a copy.

In [2]:
def min_path_sum(grid):
    rows, cols = len(grid), len(grid[0])
    dp = [[0] * cols for _ in range(rows)]
    for r in range(rows):
        for c in range(cols):
            if r == 0 and c == 0:
                dp[r][c] = grid[r][c]
            elif r == 0:
                dp[r][c] = dp[r][c - 1] + grid[r][c]
            elif c == 0:
                dp[r][c] = dp[r - 1][c] + grid[r][c]
            else:
                dp[r][c] = grid[r][c] + min(dp[r - 1][c], dp[r][c - 1])
    return dp[-1][-1]

print(min_path_sum([[1, 3, 1], [1, 5, 1], [4, 2, 1]]))   # 7

7


## 3. Triangle (bottom-up, O(n) space)

Move to adjacent numbers on the row below. Going **bottom-up**, each cell becomes `value + min(two children)`; the answer collects at the top. No special edge cases.

In [3]:
def minimum_total(triangle):
    dp = triangle[-1][:]
    for row in range(len(triangle) - 2, -1, -1):
        for i in range(len(triangle[row])):
            dp[i] = triangle[row][i] + min(dp[i], dp[i + 1])
    return dp[0]

print(minimum_total([[2], [3, 4], [6, 5, 7], [4, 1, 8, 3]]))   # 11

11


## 4. Maximal Square

`dp[r][c]` = side of the largest square of `1`s whose **bottom-right corner** is `(r, c)`.

If the cell is `1`: `dp[r][c] = 1 + min(dp[r-1][c], dp[r][c-1], dp[r-1][c-1])`. (The square is limited by its smallest neighbour.) Answer = best side squared.

In [4]:
def maximal_square(matrix):
    rows, cols = len(matrix), len(matrix[0])
    dp = [[0] * (cols + 1) for _ in range(rows + 1)]
    best = 0
    for r in range(1, rows + 1):
        for c in range(1, cols + 1):
            if matrix[r - 1][c - 1] == "1":
                dp[r][c] = 1 + min(dp[r - 1][c], dp[r][c - 1], dp[r - 1][c - 1])
                best = max(best, dp[r][c])
    return best * best

print(maximal_square([["1", "0", "1", "0", "0"], ["1", "0", "1", "1", "1"], ["1", "1", "1", "1", "1"], ["1", "0", "0", "1", "0"]]))   # 4

4


## 5. Minimum Falling Path Sum

From each cell you may go to the cell below, below-left or below-right. `dp[r][c] = matrix[r][c] + min(three cells above)`. Answer = min of the last row.

In [5]:
def min_falling_path_sum(matrix):
    n = len(matrix)
    prev = matrix[0][:]
    for r in range(1, n):
        cur = []
        for c in range(n):
            best = prev[c]
            if c > 0: best = min(best, prev[c - 1])
            if c < n - 1: best = min(best, prev[c + 1])
            cur.append(matrix[r][c] + best)
        prev = cur
    return min(prev)

print(min_falling_path_sum([[2, 1, 3], [6, 5, 4], [7, 8, 9]]))   # 13

13


## 6. Longest Increasing Path in a Matrix (memoized DFS)

Moves are in 4 directions, so there is no simple fill order. Use **DFS + memoization**: `longest(r, c)` = `1 + max(longest(neighbour))` over neighbours with a **larger** value. Because values must strictly increase, there are no cycles.

In [6]:
from functools import lru_cache

def longest_increasing_path(matrix):
    rows, cols = len(matrix), len(matrix[0])
    @lru_cache(maxsize=None)
    def dfs(r, c):
        best = 1
        for dr, dc in ((1, 0), (-1, 0), (0, 1), (0, -1)):
            nr, nc = r + dr, c + dc
            if 0 <= nr < rows and 0 <= nc < cols and matrix[nr][nc] > matrix[r][c]:
                best = max(best, 1 + dfs(nr, nc))
        return best
    return max(dfs(r, c) for r in range(rows) for c in range(cols))

print(longest_increasing_path([[9, 9, 4], [6, 6, 8], [2, 1, 1]]))   # 4

4


## 7. Dungeon Game (fill **backwards**)

A knight needs the **minimum starting health** to reach the princess (bottom-right) without health dropping to 0 or below. Going forward is hard (the path's best prefix matters). Go **backwards**: `dp[r][c]` = minimum health needed **when entering** `(r, c)`.

`dp[r][c] = max(1, min(dp[r+1][c], dp[r][c+1]) - dungeon[r][c])`.

**Lesson:** if forward DP seems to need extra state, try **reversing the direction**.

In [7]:
def calculate_minimum_hp(dungeon):
    rows, cols = len(dungeon), len(dungeon[0])
    INF = float("inf")
    dp = [[INF] * (cols + 1) for _ in range(rows + 1)]
    dp[rows][cols - 1] = dp[rows - 1][cols] = 1
    for r in range(rows - 1, -1, -1):
        for c in range(cols - 1, -1, -1):
            need = min(dp[r + 1][c], dp[r][c + 1]) - dungeon[r][c]
            dp[r][c] = max(1, need)
    return dp[0][0]

print(calculate_minimum_hp([[-2, -3, 3], [-5, -10, 1], [10, 30, -5]]))   # 7

7


## 8. Interview thinking

**Signals:** "paths in a grid", "right/down moves", "minimum/maximum sum path", "largest square".

**Say:** "`dp[r][c]` is ... It depends on the cell above and the cell to the left, so I fill row by row. O(R·C) time and O(C) space with one row."

**Common mistakes:** forgetting the first row/column, off-by-one when using a padded `(rows+1) × (cols+1)` table, resetting `dp[c]` for obstacles, wrong direction for backward DP, using DP when the move set creates cycles (use Dijkstra/BFS instead).

## Quick Revision

- Cell depends on top/left: `dp[c] += dp[c-1]`, `min`, or `max` combos.
- Maximal square: `1 + min(up, left, diag)`.
- Triangle: bottom-up.
- 4-direction increasing moves: memoized DFS.
- If forward DP needs a lot of state, go backwards (Dungeon Game).